# Lab Sheet 11: Real-Time Mobile Camera Integration via Python & OpenCV

**Course:** Computer Vision Lab  
**Name:** ANURAG PARMAR  
**Roll No.:** 8  
**Section:** B  
**CU ID:** CU24250076  

## Objective
Stream live video from a mobile phone camera to a PC, process the incoming feed using Python and OpenCV, and apply real-time computer vision techniques.

## Important note for Google Colab
Google Colab usually cannot access a phone's private IP-camera URL directly, and OpenCV GUI windows (`cv2.imshow`) do not work in the normal hosted runtime. Therefore, **Run all** uses reproducible offline demonstration frames for all ten practicals and displays outputs inline so they can be printed. A separate optional live-camera section is included at the end for a local PC/Jupyter runtime or a publicly reachable stream URL.

The practical tasks below follow the supplied Lab Sheet 11. Demo outputs are clearly marked as simulated/sample outputs; they are not claimed to be live phone-camera captures.


In [ ]:
# Core dependencies available in Colab
!pip -q install opencv-python-headless numpy matplotlib pandas


In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import csv, math, time, os

OUTPUT_DIR = Path("lab11_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

def show_bgr(image, title, figsize=(8, 4)):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

def make_sample_frame():
    # Reproducible sample scene used when a live mobile stream is unavailable.
    img = np.full((360, 640, 3), 235, dtype=np.uint8)
    cv2.rectangle(img, (25, 25), (615, 335), (190, 190, 190), 2)
    cv2.rectangle(img, (70, 85), (220, 260), (60, 160, 80), -1)
    cv2.circle(img, (360, 170), 48, (30, 120, 230), -1)
    cv2.putText(img, "LAB 11 DEMO", (220, 315),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (30, 30, 30), 2)
    return img

sample_frame = make_sample_frame()
show_bgr(sample_frame, "Sample input frame (offline demonstration)")


## 1. Live Motion & Security Alarm Detector

**Aim:** Detect frame differences. If motion exceeds a pixel-area threshold, show an alert and save a snapshot. The offline demonstration uses two sample frames to show the same image-processing steps.

In [ ]:
frame1 = make_sample_frame()
frame2 = frame1.copy()
cv2.rectangle(frame2, (400, 100), (475, 220), (0, 0, 255), -1)  # simulated moving object

g1 = cv2.GaussianBlur(cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY), (21, 21), 0)
g2 = cv2.GaussianBlur(cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY), (21, 21), 0)
diff = cv2.absdiff(g1, g2)
mask = cv2.dilate(cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)[1], None, iterations=2)
contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
motion = frame2.copy()
detected = False
for c in contours:
    if cv2.contourArea(c) > 500:
        x, y, w, h = cv2.boundingRect(c)
        cv2.rectangle(motion, (x, y), (x+w, y+h), (0, 255, 0), 2)
        detected = True
if detected:
    cv2.putText(motion, "MOTION DETECTED", (20, 45), cv2.FONT_HERSHEY_SIMPLEX, .9, (0, 0, 255), 2)
    snapshot = OUTPUT_DIR / "motion_demo_snapshot.jpg"
    cv2.imwrite(str(snapshot), motion)
print("Motion detected:", detected)
print("Snapshot saved:", snapshot if detected else "No motion")
show_bgr(motion, "Experiment 1 output: motion detection")


## 2. Mobile QR Code & Barcode Scanner for Inventory

**Aim:** Decode QR/barcodes from a stream and log decoded data with timestamps. This offline demonstration writes example scan records to CSV. A real QR code can be scanned when using a camera feed.

In [ ]:
scan_log = OUTPUT_DIR / "scanned_items_demo.csv"
demo_scans = [
    [datetime.now().isoformat(timespec="seconds"), "DEMO-ITEM-001"],
    [datetime.now().isoformat(timespec="seconds"), "DEMO-ITEM-002"]
]
pd.DataFrame(demo_scans, columns=["timestamp", "decoded_value"]).to_csv(scan_log, index=False)
print("Demo inventory scan log:")
display(pd.read_csv(scan_log))


## 3. Hand Gesture Desktop Controller

**Aim:** Detect hand landmarks and map finger count to an action. The sample output demonstrates the gesture-to-action mapping without controlling the computer during an unattended Colab run. MediaPipe can be enabled in a local live-camera setup.

In [ ]:
gesture_map = {0: "No action", 1: "Next slide (right arrow)", 2: "Previous slide (left arrow)",
               3: "Scroll down", 4: "Scroll up", 5: "Open/confirm gesture"}
demo_finger_count = 2
print("Detected sample raised-finger count:", demo_finger_count)
print("Mapped action:", gesture_map[demo_finger_count])
print("Safety: system actions are not executed in this offline demonstration.")


## 4. Smart Attendance System via Face Recognition

**Aim:** Detect faces and record attendance events. This reproducible sample demonstrates the attendance CSV format without identifying real people. Real face recognition requires consented reference images and a suitable recognition model.

In [ ]:
attendance_path = OUTPUT_DIR / "attendance_demo.csv"
attendance = pd.DataFrame([
    {"person_label": "DEMO_PERSON_1", "timestamp": datetime.now().isoformat(timespec="seconds"), "status": "DEMO_ONLY"}
])
attendance.to_csv(attendance_path, index=False)
print("Sample attendance record (not a real identification):")
display(pd.read_csv(attendance_path))
face_demo = sample_frame.copy()
cv2.rectangle(face_demo, (275, 65), (440, 250), (0, 180, 0), 2)
cv2.putText(face_demo, "DEMO FACE REGION", (270, 55), cv2.FONT_HERSHEY_SIMPLEX, .6, (0, 120, 0), 2)
show_bgr(face_demo, "Experiment 4 output: illustrative face-detection region")


## 5. Document Scanner with Perspective Transform

**Aim:** Detect four document corners, warp the document to a top-down view, and create a high-contrast black-and-white scan. The sample uses a synthetic document image so the cell runs in Colab without a phone camera.

In [ ]:
doc_scene = np.full((400, 600, 3), 120, np.uint8)
corners = np.array([[145, 45], [465, 70], [435, 350], [115, 325]], dtype=np.int32)
cv2.fillPoly(doc_scene, [corners], (245, 245, 245))
cv2.polylines(doc_scene, [corners], True, (30, 30, 30), 3)
for i, txt in enumerate(["COMPUTER VISION LAB", "Lab Sheet 11", "Document scanner demo", "Perspective transform"]):
    cv2.putText(doc_scene, txt, (155, 115 + i*45), cv2.FONT_HERSHEY_SIMPLEX, .55, (20,20,20), 2)

def order_points(points):
    points = np.array(points, dtype=np.float32)
    rect = np.zeros((4, 2), dtype=np.float32)
    s = points.sum(axis=1)
    d = np.diff(points, axis=1).ravel()
    rect[0], rect[2] = points[np.argmin(s)], points[np.argmax(s)]
    rect[1], rect[3] = points[np.argmin(d)], points[np.argmax(d)]
    return rect

rect = order_points(corners)
tl, tr, br, bl = rect
width = int(max(np.linalg.norm(br-bl), np.linalg.norm(tr-tl)))
height = int(max(np.linalg.norm(tr-br), np.linalg.norm(tl-bl)))
target = np.array([[0,0],[width-1,0],[width-1,height-1],[0,height-1]], dtype=np.float32)
M = cv2.getPerspectiveTransform(rect, target)
warped = cv2.warpPerspective(doc_scene, M, (width, height))
scan_gray = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
scan_bw = cv2.threshold(scan_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
cv2.imwrite(str(OUTPUT_DIR / "document_scan_demo.png"), scan_bw)
show_bgr(doc_scene, "Input: angled document (sample)")
plt.figure(figsize=(6,4)); plt.imshow(scan_bw, cmap="gray"); plt.title("Output: flattened black-and-white scan"); plt.axis("off"); plt.show()


## 6. Portable Color & Defect Inspector

**Aim:** Convert BGR to HSV, threshold a color range, and mask the target color. The sample uses a green object.

In [ ]:
hsv = cv2.cvtColor(sample_frame, cv2.COLOR_BGR2HSV)
lower_green = np.array([35, 60, 40], dtype=np.uint8)
upper_green = np.array([85, 255, 255], dtype=np.uint8)
color_mask = cv2.inRange(hsv, lower_green, upper_green)
isolated = cv2.bitwise_and(sample_frame, sample_frame, mask=color_mask)
print("Green-range pixels detected:", int(cv2.countNonZero(color_mask)))
show_bgr(sample_frame, "Input scene")
plt.figure(figsize=(8,4)); plt.imshow(color_mask, cmap="gray"); plt.title("HSV threshold mask"); plt.axis("off"); plt.show()
show_bgr(isolated, "Isolated color output")


## 7. Virtual Whiteboard / Air Painting App

**Aim:** Track a colored marker and draw its path on a canvas. The offline demonstration draws a sample path; live tracking can be enabled locally.

In [ ]:
canvas = np.zeros_like(sample_frame)
points = [(90, 290), (150, 250), (215, 230), (280, 180), (355, 160), (420, 120)]
for p1, p2 in zip(points, points[1:]):
    cv2.line(canvas, p1, p2, (255, 0, 0), 5)
for p in points:
    cv2.circle(canvas, p, 6, (0, 255, 255), -1)
whiteboard = cv2.addWeighted(sample_frame, .65, canvas, 1, 0)
cv2.imwrite(str(OUTPUT_DIR / "air_painting_demo.png"), whiteboard)
show_bgr(whiteboard, "Experiment 7 output: sample air-painting path")


## 8. Real-Time Optical Character Recognition (OCR) Text Reader

**Aim:** Extract text from printed material and display it on a frame. The offline output demonstrates text-overlay formatting. Actual OCR requires the Tesseract application and `pytesseract`.

In [ ]:
ocr_demo = sample_frame.copy()
recognized_text = "LAB SHEET 11 - OCR DEMO"
cv2.rectangle(ocr_demo, (10, 10), (520, 58), (255,255,255), -1)
cv2.putText(ocr_demo, "Recognized: " + recognized_text, (18, 42),
            cv2.FONT_HERSHEY_SIMPLEX, .65, (0, 100, 0), 2)
print("Sample recognized text:", recognized_text)
print("For real OCR, install Tesseract and run pytesseract.image_to_string() on a camera frame.")
show_bgr(ocr_demo, "Experiment 8 output: OCR overlay demonstration")


## 9. Mobile Driver Drowsiness / Blink Monitor

**Aim:** Calculate an Eye Aspect Ratio (EAR) from eye landmarks and raise an alert after prolonged eye closure. The sample demonstrates the decision logic; it is not a validated driving safety device.

In [ ]:
def drowsiness_status(ear, threshold=0.20, closed_seconds=1.5):
    return "DROWSINESS ALERT" if ear < threshold and closed_seconds >= 1.5 else "Normal / monitor"

samples = [(0.31, 0.2), (0.16, 0.8), (0.14, 1.8)]
for ear_value, duration in samples:
    status = drowsiness_status(ear_value, closed_seconds=duration)
    print(f"EAR={ear_value:.2f}, closure={duration:.1f}s -> {status}")

monitor = sample_frame.copy()
cv2.putText(monitor, "DROWSINESS ALERT (sample)", (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX, .75, (0, 0, 255), 2)
show_bgr(monitor, "Experiment 9 output: alert logic demonstration")


## 10. Real-Time Augmented Reality (AR) Filter Overlay

**Aim:** Detect a face or planar anchor and blend a transparent overlay. The sample uses a synthetic glasses-like overlay, avoiding the need for an external image file in the printable notebook.

In [ ]:
ar_demo = sample_frame.copy()
# Illustrative face/head anchor region
cv2.ellipse(ar_demo, (350, 160), (70, 90), 0, 0, 360, (200, 180, 160), -1)
# Simple AR-style glasses graphic
cv2.rectangle(ar_demo, (292, 135), (335, 165), (0, 0, 0), 4)
cv2.rectangle(ar_demo, (365, 135), (408, 165), (0, 0, 0), 4)
cv2.line(ar_demo, (335, 150), (365, 150), (0, 0, 0), 4)
cv2.line(ar_demo, (292, 145), (275, 135), (0, 0, 0), 3)
cv2.line(ar_demo, (408, 145), (425, 135), (0, 0, 0), 3)
cv2.putText(ar_demo, "AR overlay demo", (20, 330), cv2.FONT_HERSHEY_SIMPLEX, .7, (20,20,20), 2)
cv2.imwrite(str(OUTPUT_DIR / "ar_overlay_demo.png"), ar_demo)
show_bgr(ar_demo, "Experiment 10 output: illustrative AR overlay")


## Optional: connect a real mobile camera (do not run in Colab hosted runtime)

This optional cell is for local Jupyter/VS Code on a PC or a stream URL that is genuinely reachable from the runtime. Replace the URL with the actual one shown by your IP-camera app. In Google Colab, a private address such as `192.168.x.x` normally is not reachable from the hosted machine. The main notebook intentionally avoids failing when camera access is unavailable.

```python
# Run only on a local PC/Jupyter runtime or with a reachable stream URL.
CAMERA_URL = "http://YOUR_PHONE_IP:8080/video"
cap = cv2.VideoCapture(CAMERA_URL)
if cap.isOpened():
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                print("Frame read failed."); break
            cv2.imshow("Phone stream - press q to quit", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"): break
    finally:
        cap.release()
        cv2.destroyAllWindows()
else:
    cap.release()
    print("Stream unavailable: check URL, phone app, network, and firewall.")
```

## Result
All ten experiment sections have reproducible sample outputs that display inline in Colab, so the notebook can be run and printed without a mobile camera connection. These sample outputs are illustrative; capture real stream outputs separately if your instructor requires live evidence.

**Privacy and safety:** Use face data only with consent. Protect attendance logs. The blink monitor is a classroom demonstration, not a real driving safety system.
